# Selected Experiment Metrics Summary

This notebook summarizes the selected CE1 experiment results from `results/` and local `wandb/` folders.

The table is sorted by `primary_rank_score`, where lower is better:

- 50% load-tracking CV-RMSE rank
- 25% absolute NMBE rank
- 25% thermal-comfort exceedance rank


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd()
if not (REPO_DIR / 'results').exists():
    REPO_DIR = REPO_DIR.parent

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from tools.summarize_selected_experiment_metrics import (
    MARKDOWN_COLUMNS,
    SELECTED_EXPERIMENTS,
    SUMMARY_COLUMNS,
    _add_primary_ranking,
    _build_row,
    _write_markdown_table,
)

OUTPUT_ROOT = REPO_DIR / 'experiment_metric_summary'
WANDB_ROOT = REPO_DIR / 'wandb'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Repo: {REPO_DIR}')
print(f'Output: {OUTPUT_ROOT}')


## Selected Experiments

In [ ]:
pd.DataFrame({'experiment': SELECTED_EXPERIMENTS})

## Build Summary Table

In [ ]:
rows = [_build_row(experiment, WANDB_ROOT) for experiment in SELECTED_EXPERIMENTS]
df = _add_primary_ranking(pd.DataFrame(rows))

all_columns = SUMMARY_COLUMNS + [column for column in df.columns if column not in SUMMARY_COLUMNS]
df = df.loc[:, [column for column in all_columns if column in df.columns]]

full_csv = OUTPUT_ROOT / 'selected_experiment_metrics_full.csv'
report_csv = OUTPUT_ROOT / 'selected_experiment_metrics_report_table.csv'
markdown_path = OUTPUT_ROOT / 'selected_experiment_metrics_report_table.md'

df.to_csv(full_csv, index=False)
df.loc[:, [column for column in SUMMARY_COLUMNS if column in df.columns]].to_csv(report_csv, index=False)
_write_markdown_table(markdown_path, df, MARKDOWN_COLUMNS)

print(f'Full metrics table -> {full_csv}')
print(f'Report table -> {report_csv}')
print(f'Markdown table -> {markdown_path}')


## Report Table

In [ ]:
report_columns = [column for column in MARKDOWN_COLUMNS if column in df.columns]
display(df[report_columns])

## Primary Metrics Plots

In [ ]:
plot_df = df.sort_values('primary_rank').copy()
labels = plot_df['experiment'].str.replace('_vt_500_final', '', regex=False)
labels = labels.str.replace('_vt_42_final', '', regex=False)
labels = labels.str.replace('_vt_50_final', '', regex=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)

metrics = [
    ('primary_load_cv_rmse_pct', 'Load Tracking CV-RMSE (%)'),
    ('primary_abs_nmbe_pct', '|NMBE| (%)'),
    ('primary_comfort_exceedance_pct', 'Comfort Exceedance (%)'),
]

for ax, (column, title) in zip(axes, metrics):
    ax.barh(labels, plot_df[column], color='#4e79a7')
    ax.invert_yaxis()
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.25)

fig.suptitle('Selected Experiments - Primary Metrics')
fig.tight_layout()

plot_path = OUTPUT_ROOT / 'selected_experiment_primary_metrics.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
print(f'Plot -> {plot_path}')
plt.show()


## Secondary Metrics Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)

secondary_metrics = [
    ('kpi_electricity_consumption', 'Electricity Consumption KPI'),
    ('kpi_ramping', 'Ramping KPI'),
    ('secondary_peak_demand_change_pct', 'Peak Demand Change (%)'),
]

for ax, (column, title) in zip(axes, secondary_metrics):
    ax.barh(labels, plot_df[column], color='#59a14f')
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.25)

fig.suptitle('Selected Experiments - Secondary Metrics')
fig.tight_layout()

plot_path = OUTPUT_ROOT / 'selected_experiment_secondary_metrics.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
print(f'Plot -> {plot_path}')
plt.show()


## W&B Match Check

In [ ]:
wandb_check_columns = ['experiment', 'wandb_run_id', 'wandb_started_at', 'wandb_program', 'wandb_run_dir']
display(df[[column for column in wandb_check_columns if column in df.columns]])